# Genome-scale metabolic models

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:
1. Navigate the metabolites, reactions and genes of a cobrapy model and look up their properties.
2. Interpret compartments, formulas, annotations and gene-protein-reaction (GPR) rules.
3. Relate the model objects to the stoichiometric matrix $S$ and to the FBA linear program.
4. Search a model for metabolites or reactions by name.
5. Recognize an infeasible optimization and explain why it happened.

**Estimated time:** 45 min.

**Prerequisites:** notebook 01.

### Background

A cobrapy `Model` has three main lists (`DictList`s): `model.metabolites`, `model.reactions` and `model.genes`. They are connected:

- A **metabolite** belongs to one **compartment** (cytosol `c`, periplasm `p`, extracellular `e` in *E. coli*). The same chemical in two compartments is two different metabolites, e.g. `glc__D_e` and `glc__D_p`, linked by transport reactions.
- A **reaction** has stoichiometric coefficients for its metabolites (negative for substrates, positive for products) and **flux bounds** (`lower_bound`, `upper_bound`, in mmol gDW⁻¹ h⁻¹). An irreversible reaction has `lower_bound = 0`. **Exchange reactions** (`EX_...`) move metabolites across the system boundary and define the medium.
- A **gene** is linked to reactions via **GPR rules**, Boolean expressions such as `b1723 or b3916` (isozymes: either gene is enough) or `b3731 and b3732` (subunits of one complex: all are needed). GPRs let us translate a gene knockout into the reactions it disables.

The stoichiometric coefficients of all reactions form the matrix $S$ (metabolites × reactions). FBA imposes $S v = 0$: for every internal metabolite, production equals consumption ([Orth et al., 2010](https://doi.org/10.1038/nbt.1614)). The **objective** (usually a biomass reaction that drains amino acids, nucleotides, lipids, cofactors etc. in the ratio found in dry cell mass) is what the solver maximizes.

The **ATP maintenance reaction** (`ATPM`) forces a minimal flux of ATP hydrolysis. It represents the energy the cell spends on things not included in the biomass reaction (turnover, maintaining gradients). This is *non-growth-associated* maintenance; the growth-associated part is included in the biomass reaction.

## Preparation

Load *i*JO1366 again.

In [ ]:
from cobra.io import read_sbml_model
model = read_sbml_model('data/iJO1366.xml.gz')

## Model content

### Metabolites

The model contains a list of metabolites. Here are the first ten.

In [ ]:
model.metabolites[0:10]

There are 1805 metabolites in the model.

In [ ]:
len(model.metabolites)

One can access a specific metabolite using dot notation.

In [ ]:
model.metabolites.g3p_c

<div class="alert alert-warning">

**Warning:** One cannot use dot notation to access metabolites, reactions, or genes if their identifiers do not resemble proper Python variable names.

</div>

In [ ]:
# model.metabolites.10fthf_c  # uncomment to see the SyntaxError: identifiers can't start with a digit

<div class="alert alert-success">

**Solution:** Use the method `get_by_id` instead!

</div>

In [ ]:
model.metabolites.get_by_id('10fthf_c')

Metabolites are associated with compartments in the cell. Glyceraldehyde 3-phosphate (`g3p_c`) is associated with the `c` (Cytosol) compartment.

In [ ]:
model.metabolites.g3p_c.compartment

The _E. coli_ model has three compartments.

In [ ]:
model.compartments

Some metabolites (like Glucose for example) can be associated with multiple compartments.

In [ ]:
model.metabolites.glc__D_c.compartment

In [ ]:
model.metabolites.glc__D_p.compartment

The full name of the metabolite is available via the `.name` attribute.

In [ ]:
model.metabolites.glc__D_c.name

One can look up the molecular formula of a metabolite, here glyceraldehyde 3-phosphate. Formulas (and charges) are used to check that reactions are mass- and charge-balanced.

In [ ]:
model.metabolites.g3p_c.formula

The `.elements` attribute returns a dictionary representation of the formula.

In [ ]:
model.metabolites.g3p_c.elements

Furthermore, one can look up the molecular weight of a metabolite.

In [ ]:
model.metabolites.g3p_c.formula_weight

One can gather additional information (like references to external databases) about the metabolite through the `annotation` attribute. Annotations are how you map model identifiers to KEGG, ChEBI, MetaNetX etc., for example when integrating omics data.

In [ ]:
model.metabolites.g3p_c.annotation

One can use these annotations to look up the compound on [KEGG](https://www.genome.jp/dbget-bin/www_bget?cpd:C00118) for example.

Metabolites are not isolated things. They participate in reactions as substrates and products.

In [ ]:
model.metabolites.g3p_c.reactions

### Reactions

The model contains a list of reactions. Here are the first 10 of them.

In [ ]:
model.reactions[0:10]

There are 2583 reactions in the model.

In [ ]:
len(model.reactions)

Let's take a closer look at the reactions associated with Glyceraldehyde 3-phosphate (`g3p`).

In [ ]:
for reaction in model.metabolites.g3p_c.reactions:
    print(reaction.id, reaction, reaction.name)

One of the reactions in this list is glyceraldehyde-3-phosphate dehydrogenase (GAPD), the glycolytic step that oxidizes G3P and produces NADH.

In [ ]:
model.reactions.GAPD.name

## Objective

The objective is a linear expression over reaction fluxes. Printing it shows that the solver maximizes the flux through the core biomass reaction (both the forward and reverse variables of the reaction appear because cobrapy represents each reaction as two non-negative variables).

In [ ]:
print(model.objective)

## ATP maintenance requirement

Note the lower bound of `ATPM`: every solution must hydrolyse at least this much ATP (mmol gDW⁻¹ h⁻¹) for maintenance.

In [ ]:
model.reactions.ATPM

## The math (scary!)

This is the beginning of the actual optimization problem that cobrapy hands to the solver: one constraint per metabolite (the rows of $S v = 0$) and bounds per variable. You rarely need to look at it, but it shows that the model *is* just a linear program.

In [ ]:
print(model.solver.__str__()[0:1550])

## Gene-Protein-Reaction associations

Glyceraldehyde-3-phosphate dehydrogenase is associated with a single gene.

In [ ]:
model.reactions.GAPD.gene_reaction_rule

Phosphofructokinase (PFK) on the other hand is associated with two isozymes (`or`): knocking out one of the two genes does not block the reaction.

In [ ]:
model.reactions.PFK.gene_reaction_rule

One can display the gene names (typical 4 letter gene codes) instead of the identifiers (Blattner numbers in this case).

In [ ]:
model.reactions.PFK.gene_name_reaction_rule

Here a very complicated gene to reaction mapping (ATP synthase): a multi-subunit complex (`and`), where some subunits can be provided by alternative genes.

In [ ]:
model.reactions.ATPS4rpp.gene_name_reaction_rule

## Finding things

One can use `.query('search term', 'attribute_to_search_in')` to search in model metabolites, reactions, and genes. For example, one can search metabolites that contain the term _glucose_ in their name.

In [ ]:
for metabolite in model.metabolites.query('glucose', 'name'):
    print(metabolite.name)

## The stoichiometric matrix S

In [ ]:
from cobra.util import create_stoichiometric_matrix
s = create_stoichiometric_matrix(model)
s

This is how the stoichiometric matrix $S$ looks when visualized as a matrix plot (each dot is a non-zero coefficient). It is very sparse: most reactions involve only a handful of the 1805 metabolites. Horizontal lines are highly connected metabolites such as H⁺, H₂O, ATP and NAD(H).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.spy(s, precision=0.01, markersize=.1)

## Simulating models

In [ ]:
model.optimize()

Sometimes a solution cannot be found. For example, setting the lower bound of the objective function to a very high value that the model cannot achieve will trigger a warning when trying to optimize the model. Parameters reported from an infeasible model are not meaningful to interpret (except in rare occasions when you may want to figure out why a model is infeasible).

In [ ]:
infeasible_model = model.copy()
infeasible_model.reactions.BIOMASS_Ec_iJO1366_core_53p95M.bounds = 100000, 100000
sol = infeasible_model.optimize()
sol.status

`sol.status` is `infeasible`: no flux vector satisfies all constraints at the same time (the model cannot grow at 100000 h⁻¹). Always check `solution.status` (or use `model.slim_optimize(error_value=...)`) before interpreting results, especially in loops over many conditions.

## References

- Orth, Thiele, Palsson (2010). What is flux balance analysis? *Nature Biotechnology* 28:245–248. [doi.org/10.1038/nbt.1614](https://doi.org/10.1038/nbt.1614)